# Tutorial 03c: Autoencoders with CNNs
Prof Ivan Olier

## Introduction

This tutorial extends the previous autoencoder work by replacing dense layers with convolutional layers. The aim is to show how convolutional autoencoders can reconstruct image data while preserving spatial structure. We first examine two common upsampling mechanisms used in decoders, then implement a convolutional autoencoder for MNIST digits and inspect its bottleneck representation.


We begin with the standard numerical and plotting libraries used throughout the notebook. `matplotlib` will be used to inspect matrices, learning curves, reconstructions, and latent representations.


In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
%matplotlib inline

The convolutional autoencoder will be built using the Keras Functional API. This is useful here because the same input tensor will later be reused to define both the full autoencoder and the encoder-only model.


In [ ]:
from tensorflow.keras.backend import clear_session
from tensorflow.keras.layers import Input, Dense, Conv2D, MaxPooling2D, UpSampling2D
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model

This helper function plots the training and validation losses. It provides a quick visual check of convergence and possible overfitting during training.


In [ ]:
def plot_history(history):
    plt.figure()
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend(['train', 'validation'])
    return;

Visual inspection is particularly important for autoencoders. The next helper function shows original images in the first row and their reconstructions in the second row, making it easier to judge whether the model has retained the relevant image structure.


In [ ]:
def plot_digits(X_true, X_pred):
    n = X_pred.shape[0]  # how many digits we will display
    plt.figure(figsize=(20, 4))
    for i in range(n):
        # display original
        ax = plt.subplot(2, n, i + 1)
        plt.imshow(X_true[i].reshape(28, 28))
        plt.gray()
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
        # display reconstruction
        ax = plt.subplot(2, n, i + 1 + n)
        plt.imshow(X_pred[i].reshape(28, 28))
        plt.gray()
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
    plt.show()


## Upsampling

Decoders often need to increase spatial resolution after an encoder has compressed the input. Two common approaches are fixed upsampling, such as nearest-neighbour or bilinear interpolation, and learnable upsampling, such as transposed convolution.

### Scaling


Before building the full autoencoder, we first examine how upsampling works. The small random matrix below gives a simple numerical example that is easier to understand than a full image.


In [ ]:
import tensorflow as tf

# Create a random 3x3 matrix
x = np.random.randint(0, 10, size=(1, 3, 3, 1))
print("Original matrix:")
print(tf.squeeze(x))

Nearest-neighbour upsampling increases the spatial dimensions by copying each value into a larger local block. This is simple and fast, but it can create blocky outputs when applied to images.


In [ ]:
# Apply UpSampling2D with size (3, 3)
y = UpSampling2D(size=(3, 3))(x)

print("\nUpsampled matrix:")
print(tf.squeeze(y))

Bilinear interpolation also increases the spatial dimensions, but it estimates new values using neighbouring values. In image applications, this typically produces smoother upsampled outputs than nearest-neighbour interpolation.


In [ ]:
# Apply UpSampling2D with size (3, 3)
y = UpSampling2D(size=(3, 3), interpolation='bilinear')(x)

print("\nUpsampled matrix:")
print(tf.squeeze(y))

### Transposed convolution


A transposed convolution is different from fixed interpolation because its weights are learned during training. This makes it useful in decoders, image generation, and segmentation models, although the output below is not meaningful yet because the layer has not been trained.


In [ ]:
from tensorflow.keras.layers import Conv2DTranspose

# Cast the input tensor to float32
xf = tf.cast(x, tf.float32)
# Apply Conv2DTranspose to achieve similar upsampling effect
# Stride of (3, 3) and kernel size of (3, 3) with padding='same' will upsample by 3
y = Conv2DTranspose(filters=1, kernel_size=(3, 3), strides=(3, 3), padding='same')(xf)

print("\nTransposed convolution output:")
print(tf.squeeze(y))

## Convolutional autoencoder

A convolutional autoencoder applies the same encoder-decoder idea as a dense autoencoder, but it is better suited to images because it works directly with local spatial patterns. Convolutional layers learn filters over neighbourhoods of pixels, while pooling and upsampling control the spatial resolution.


We now move to the convolutional autoencoder example. MNIST images are loaded and reshaped from 28 × 28 matrices into 28 × 28 × 1 tensors, where the final dimension represents the single greyscale channel expected by convolutional layers.


In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.reshape((X_train.shape[0], 28, 28, 1))
X_test = X_test.reshape((X_test.shape[0], 28, 28, 1))

The pixel intensities are converted to floating point values and scaled to the range \([0,1]\). This scaling matches the sigmoid activation used in the final reconstruction layer.


In [ ]:
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
X_train /= 255
X_test /= 255

The architecture has an encoder-decoder structure. The encoder uses convolution and max-pooling to reduce the image to a compact bottleneck. The decoder then uses convolution and upsampling to reconstruct the original image size.

The bottleneck has spatial dimensions rather than a single vector. This means it can preserve local visual patterns, such as strokes and edges, while still compressing the image.


In [ ]:
input_img = Input(shape=(28, 28, 1))
x = Conv2D(16, (3, 3), activation='relu', padding='same')(input_img)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(8, (3, 3), activation='relu', padding='same')(x)
bottleneck = MaxPooling2D((2, 2), padding='same')(x)

x = Conv2D(8, (3, 3), activation='relu', padding='same')(bottleneck)
x = UpSampling2D((2, 2))(x)
x = Conv2D(16, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)
decoded = Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

model4 = Model(input_img, decoded)
model4.summary()

The model is trained as a reconstruction model: the input image is also the target image. Binary cross-entropy is appropriate here because the pixels have been scaled to \([0,1]\) and the output layer uses a sigmoid activation.


In [ ]:
model4.compile(loss='binary_crossentropy', optimizer='adam')

The autoencoder is trained to reproduce its input images. A validation split is used so that the reconstruction loss can be monitored on data not used for updating the weights.


In [ ]:
history = model4.fit(X_train, X_train,
          batch_size=256, epochs=15,
          verbose=1,
          validation_split=0.2)

After training, the model is applied to the test set. Comparing the original and reconstructed digits shows how much information the bottleneck has retained.


In [ ]:
X_pred = model4.predict(X_test)
plot_digits(X_test[:10], X_pred[:10])

The full autoencoder contains both the encoder and decoder. To inspect the learned representation directly, we define a second model that shares the same input but stops at the bottleneck layer.


In [ ]:
encoder_model = Model(inputs=input_img,
                     outputs=bottleneck)
encoder_model.summary()

The encoder-only model converts each test image into its bottleneck representation. These encoded images are lower-dimensional feature maps learned from the reconstruction task.


In [ ]:
encoded_img = encoder_model.predict(X_test)

The bottleneck activations are visualised below. They should not be interpreted as normal images; they are compact feature maps that summarise information the decoder needs to reconstruct each digit.


In [ ]:
n = 10
plt.figure(figsize=(20, 8))
for i in range(n):
    ax = plt.subplot(1, n, i+1)
    plt.imshow(encoded_img[i].reshape(7, 7 * 8).T)
    plt.gray()
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
plt.show()

## Exercise

1. How does this bottleneck differ from the bottlenecks obtained in the previous tutorial? What might the bottleneck feature maps represent?
2. Rebuild the model using a different set of hyperparameters. Do the reconstructed images or bottleneck plots change? Comment on the differences.
3. Implement a similar CNN autoencoder using the Fashion-MNIST data. Experiment with different hyperparameters until you obtain an acceptable reconstruction. Which hyperparameters had the greatest influence?
